In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import add_messages
from langchain_aws.chat_models import ChatBedrock
import boto3
from dotenv import load_dotenv
import os

In [2]:
load_dotenv(override=True)

True

In [3]:
from botocore.credentials import RefreshableCredentials
from botocore.session import get_session

ROLE_ARN = os.getenv('ROLE_ARN')
REGION = os.getenv('AWS_REGION')

ROLE_SESSION_NAME = "bedrock-session"


sts = boto3.client("sts")
 
response = sts.assume_role(
    RoleArn=ROLE_ARN,
    RoleSessionName="bedrock-session"
)
 
creds = response["Credentials"]
 
session = boto3.Session(
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
    region_name=REGION
)

bedrock_client = session.client(
    service_name="bedrock-runtime",
    region_name=REGION
)


provider = os.getenv("PROVIDER")
model_id = os.getenv("MODEL_ID")
model_kwargs = {
    "max_tokens":1000,
    "top_p": 0.9,
}

llm = ChatBedrock(
        client=bedrock_client,
        provider=provider,
        model_id=model_id,  
        model_kwargs=model_kwargs,
        streaming=False
    )

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:

graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': '# 🍕 Pizza Joke\n\nWhy did the pizza maker go to the bank?\n\nBecause he wanted to get his **dough**! \n\n---\n\n*Bonus joke:* What did the pizza say to the topping?\n\n"You\'re on a **roll**... I mean, a *crust*!" 😄',
 'explanation': '# Joke Explanation\n\n## Main Joke: "Why did the pizza maker go to the bank?"\n\nThis joke works through **wordplay/pun** using a double meaning:\n\n- **"Dough"** has two meanings:\n  1. **Pizza dough** - the literal ingredient used to make pizza\n  2. **Dough** - slang for money\n\nThe humor comes from the setup making you think about pizza (the pizza maker context), but then the punchline plays with the financial meaning. A pizza maker going to a bank would normally be going for money-related reasons, but here it\'s a clever twist that connects back to their profession.\n\n---\n\n## Bonus Joke Explanation\n\nThis one is slightly more forced but uses a similar structure:\n\n- **"Roll"** has multiple meanings:\n  1. A bread ro

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': '# 🍕 Pizza Joke\n\nWhy did the pizza maker go to the bank?\n\nBecause he wanted to get his **dough**! \n\n---\n\n*Bonus joke:* What did the pizza say to the topping?\n\n"You\'re on a **roll**... I mean, a *crust*!" 😄', 'explanation': '# Joke Explanation\n\n## Main Joke: "Why did the pizza maker go to the bank?"\n\nThis joke works through **wordplay/pun** using a double meaning:\n\n- **"Dough"** has two meanings:\n  1. **Pizza dough** - the literal ingredient used to make pizza\n  2. **Dough** - slang for money\n\nThe humor comes from the setup making you think about pizza (the pizza maker context), but then the punchline plays with the financial meaning. A pizza maker going to a bank would normally be going for money-related reasons, but here it\'s a clever twist that connects back to their profession.\n\n---\n\n## Bonus Joke Explanation\n\nThis one is slightly more forced but uses a similar structure:\n\n- **"Roll"** has multiple meaning

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': '# 🍕 Pizza Joke\n\nWhy did the pizza maker go to the bank?\n\nBecause he wanted to get his **dough**! \n\n---\n\n*Bonus joke:* What did the pizza say to the topping?\n\n"You\'re on a **roll**... I mean, a *crust*!" 😄', 'explanation': '# Joke Explanation\n\n## Main Joke: "Why did the pizza maker go to the bank?"\n\nThis joke works through **wordplay/pun** using a double meaning:\n\n- **"Dough"** has two meanings:\n  1. **Pizza dough** - the literal ingredient used to make pizza\n  2. **Dough** - slang for money\n\nThe humor comes from the setup making you think about pizza (the pizza maker context), but then the punchline plays with the financial meaning. A pizza maker going to a bank would normally be going for money-related reasons, but here it\'s a clever twist that connects back to their profession.\n\n---\n\n## Bonus Joke Explanation\n\nThis one is slightly more forced but uses a similar structure:\n\n- **"Roll"** has multiple meanin

In [11]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': '# 🍝 Pasta Joke\n\nWhy did the pasta go to the doctor?\n\nBecause it was feeling **rotini**! \n\n---\n\n*Alternative version:* It had **penne**-cillin deficiency! 😄',
 'explanation': '# Explanation of the Pasta Joke\n\n## Main Joke: "Rotini"\n\nThis is a **pun** based on homophone wordplay:\n\n- **"Rotini"** (the spiral-shaped pasta) sounds like **"rotten-y"** or **"rotten"**\n- The joke implies the pasta was sick/unwell, so it felt "rotten"\n- By saying it felt "rotini," it\'s a clever double meaning—both literally the pasta type AND describing the pasta\'s condition\n\n**Why it works:** The setup ("feeling rotini") sounds like natural speech about feeling unwell, but the punchline reveals the pasta pun.\n\n---\n\n## Alternative Joke: "Penne-cillin"\n\nThis uses a similar pun structure:\n\n- **"Penne"** (tube-shaped pasta) sounds like **"penny"**\n- Combined with **"cillin"** (from penicillin, an antibiotic)\n- Creates **"penne-cillin"** = a fake medicine n

In [12]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': '# 🍕 Pizza Joke\n\nWhy did the pizza maker go to the bank?\n\nBecause he wanted to get his **dough**! \n\n---\n\n*Bonus joke:* What did the pizza say to the topping?\n\n"You\'re on a **roll**... I mean, a *crust*!" 😄', 'explanation': '# Joke Explanation\n\n## Main Joke: "Why did the pizza maker go to the bank?"\n\nThis joke works through **wordplay/pun** using a double meaning:\n\n- **"Dough"** has two meanings:\n  1. **Pizza dough** - the literal ingredient used to make pizza\n  2. **Dough** - slang for money\n\nThe humor comes from the setup making you think about pizza (the pizza maker context), but then the punchline plays with the financial meaning. A pizza maker going to a bank would normally be going for money-related reasons, but here it\'s a clever twist that connects back to their profession.\n\n---\n\n## Bonus Joke Explanation\n\nThis one is slightly more forced but uses a similar structure:\n\n- **"Roll"** has multiple meaning

In [13]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': '# 🍕 Pizza Joke\n\nWhy did the pizza maker go to the bank?\n\nBecause he wanted to get his **dough**! \n\n---\n\n*Bonus joke:* What did the pizza say to the topping?\n\n"You\'re on a **roll**... I mean, a *crust*!" 😄', 'explanation': '# Joke Explanation\n\n## Main Joke: "Why did the pizza maker go to the bank?"\n\nThis joke works through **wordplay/pun** using a double meaning:\n\n- **"Dough"** has two meanings:\n  1. **Pizza dough** - the literal ingredient used to make pizza\n  2. **Dough** - slang for money\n\nThe humor comes from the setup making you think about pizza (the pizza maker context), but then the punchline plays with the financial meaning. A pizza maker going to a bank would normally be going for money-related reasons, but here it\'s a clever twist that connects back to their profession.\n\n---\n\n## Bonus Joke Explanation\n\nThis one is slightly more forced but uses a similar structure:\n\n- **"Roll"** has multiple meanin

## Time travel

In [16]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f16ba33-4cfe-6da4-8000-291fe838a9c9"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f16ba33-4cfe-6da4-8000-291fe838a9c9'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-06-19T05:53:33.507933+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f16ba33-4ce7-639a-bfff-bd62d8174e5f'}}, tasks=(PregelTask(id='65754b60-193c-5bcd-faa7-6dfa8e0874e3', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': '# 🍕 Pizza Joke\n\nWhy did the pizza maker go to the bank?\n\nBecause he wanted to get his **dough**! \n\n---\n\n*Bonus joke:* What did the pizza say to the topping?\n\n"You\'re on a **roll**... I mean, a *crust*!" 😄'}),), interrupts=())

In [18]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f16ba33-4cfe-6da4-8000-291fe838a9c9"}})

{'topic': 'pizza',
 'joke': "# 🍕 Pizza Joke\n\nWhy did the pizza maker go to the bank?\n\nBecause he needed to make some **dough**! 🥖😄\n\n---\n\n*Alternative version:* What do you call a pizza that's a detective? \n\n**Pizza-case** (Pizza case)! 🔍",
 'explanation': '# Explanation of the Pizza Jokes\n\n## Main Joke: "Why did the pizza maker go to the bank?"\n\nThis is a **pun** based on the double meaning of the word "dough":\n\n1. **Literal meaning**: Dough is the flour-based mixture used to make pizza crust\n2. **Slang meaning**: "Dough" is slang for money\n\nThe joke works because pizza makers literally work with dough every day, but the punchline plays on the unexpected twist that the pizza maker needs dough in the *financial* sense (money at a bank) rather than the *culinary* sense. This wordplay creates humor through misdirection—you expect the answer to be about pizza-making, but it pivots to a money joke.\n\n---\n\n## Alternative Joke: "What do you call a pizza that\'s a detecti

In [19]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "# 🍕 Pizza Joke\n\nWhy did the pizza maker go to the bank?\n\nBecause he needed to make some **dough**! 🥖😄\n\n---\n\n*Alternative version:* What do you call a pizza that's a detective? \n\n**Pizza-case** (Pizza case)! 🔍", 'explanation': '# Explanation of the Pizza Jokes\n\n## Main Joke: "Why did the pizza maker go to the bank?"\n\nThis is a **pun** based on the double meaning of the word "dough":\n\n1. **Literal meaning**: Dough is the flour-based mixture used to make pizza crust\n2. **Slang meaning**: "Dough" is slang for money\n\nThe joke works because pizza makers literally work with dough every day, but the punchline plays on the unexpected twist that the pizza maker needs dough in the *financial* sense (money at a bank) rather than the *culinary* sense. This wordplay creates humor through misdirection—you expect the answer to be about pizza-making, but it pivots to a money joke.\n\n---\n\n## Alternative Joke: "What do you call a piz

In [20]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f16ba8f-babd-63f0-8000-56f3754cab3b'}}

In [21]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f16ba8f-babd-63f0-8000-56f3754cab3b'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-06-19T06:34:54.621592+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='a90d4ea0-41fa-09f2-227e-d0b14261f538', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "# 🍕 Pizza Joke\n\nWhy did the pizza maker go to the bank?\n\nBecause he needed to make some **dough**! 🥖😄\n\n---\n\n*Alternative version:* What do you call a pizza that's a detective? \n\n**Pizza-case** (Pizza case)! 🔍", 'explanation': '# Explanation of the Pizza Jokes\n\n## Main Joke: "Why did the pizza maker go to the bank?"\n

In [23]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f16ba8f-65a6-635b-8002-7c8d067d06e0"}})

{'topic': 'pizza',
 'joke': "# 🍕 Pizza Joke\n\nWhy did the pizza maker go to the bank?\n\nBecause he needed to make some **dough**! 🥖😄\n\n---\n\n*Alternative version:* What do you call a pizza that's a detective? \n\n**Pizza-case** (Pizza case)! 🔍",
 'explanation': '# Explanation of the Pizza Jokes\n\n## Main Joke: Why did the pizza maker go to the bank?\n\nThis is a **pun** based on the double meaning of the word "dough":\n\n1. **Literal meaning**: Dough is the basic ingredient used to make pizza (flour, water, yeast, etc.)\n2. **Slang meaning**: "Dough" is informal slang for money\n\nThe joke plays on our expectations—we expect the punchline to be about pizza ingredients, but it cleverly switches to a financial context (going to a bank). The humor comes from this unexpected twist combined with the wordplay.\n\n---\n\n## Alternative Joke: What do you call a pizza that\'s a detective?\n\nThis is a **pun based on pronunciation**:\n\n- **"Pizza-case"** sounds like **"piece-a-case"** (a c

In [24]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "# 🍕 Pizza Joke\n\nWhy did the pizza maker go to the bank?\n\nBecause he needed to make some **dough**! 🥖😄\n\n---\n\n*Alternative version:* What do you call a pizza that's a detective? \n\n**Pizza-case** (Pizza case)! 🔍", 'explanation': '# Explanation of the Pizza Jokes\n\n## Main Joke: Why did the pizza maker go to the bank?\n\nThis is a **pun** based on the double meaning of the word "dough":\n\n1. **Literal meaning**: Dough is the basic ingredient used to make pizza (flour, water, yeast, etc.)\n2. **Slang meaning**: "Dough" is informal slang for money\n\nThe joke plays on our expectations—we expect the punchline to be about pizza ingredients, but it cleverly switches to a financial context (going to a bank). The humor comes from this unexpected twist combined with the wordplay.\n\n---\n\n## Alternative Joke: What do you call a pizza that\'s a detective?\n\nThis is a **pun based on pronunciation**:\n\n- **"Pizza-case"** sounds like **"

## fault tolerance

In [1]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [2]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [3]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [4]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))